# LAB 10 — K-Means Clustering
## Finding Customer Segments, Choosing K and Avoiding the Traps on Real Data

## Student Information
- **Name:**
- **Student ID:**
- **Section:**
- **GitHub:**
- **Kaggle:**
- **Dataset:** Mall Customers (Kaggle) + Wine (UCI)
- **Dataset Source:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python and https://archive.ics.uci.edu/dataset/109/wine

### Before You Start
- You studied K-Means in the lecture. This notebook does not teach it again; it makes you use it, test it and question it.
- There is **no target column**: no train/test split and no accuracy. Judge results by distances, silhouette, plots and common sense.
- Use `SEED = 42`. Write an interpretation under every table and plot. Place `Mall_Customers.csv` in the same folder as this notebook.
- `from sklearn.datasets import load_wine` loads the Wine data (no download needed).


---

## Part 0 — Setup and First Look (10 minutes)

Today there is no target column. The Mall Customers data has no 'right answer' for each customer, so we do not split into train and test and we cannot use accuracy. You will find groups with K-Means (your lecture) and judge them by distance, silhouette and common sense.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Unsupervised learning | Learning from data that has no labels. The goal is to find structure (groups), not to predict a column. | `No y, no train_test_split.` |
| Cluster | A group of points that are close to each other and far from the other groups. | `model.labels_ gives the cluster number of every row.` |
| Feature | One column used to describe each row (here Age, Income, Spending). | `df[['Income','Spending']].values` |


**First cell:** import everything you need (NumPy, Pandas, Matplotlib, KMeans, StandardScaler, PCA, load_wine, silhouette_score, silhouette_samples, adjusted_rand_score) and set `SEED = 42`.

In [1]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

SEED = 42

### Task 0.1 — Load and check the data

Load Mall_Customers.csv and understand it in one cell.

1. Load the CSV. Rename 'Annual Income (k$)' to Income and 'Spending Score (1-100)' to Spending (shorter names).
2. Print the shape, the number of missing values, the number of duplicate rows and the count of each Gender.
3. Show describe() for Age, Income and Spending.


In [2]:
# YOUR CODE HERE
df = pd.read_csv('Mall_Customers.csv')

df = df.rename(columns={
    'Genre': 'Gender',
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'Spending'
})

print('Shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print('\nGender counts:')
print(df['Gender'].value_counts())
print('\nDescriptive statistics:')
display(df[['Age', 'Income', 'Spending']].describe())

Shape: (200, 5)
Missing values: 0
Duplicate rows: 0

Gender counts:
Gender
Female    112
Male       88
Name: count, dtype: int64

Descriptive statistics:


,Age,Income,Spending
count,200.000000,200.000000,200.000000
mean,38.850000,60.560000,50.200000
std,13.969007,26.264721,25.823522
min,18.000000,15.000000,1.000000
25%,28.750000,41.500000,34.750000
50%,36.000000,61.500000,50.000000
75%,49.000000,78.000000,73.000000
max,70.000000,137.000000,99.000000


### Task 0.2 — Look at the data before clustering

Use your eyes first: draw three scatter plots side by side.

1. Draw Spending vs Income, Spending vs Age and Income vs Age (one row of three plots, axis labels and titles).


In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df['Income'], df['Spending'])
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score')
axes[0].set_title('Spending vs Income')

axes[1].scatter(df['Age'], df['Spending'])
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Spending vs Age')

axes[2].scatter(df['Age'], df['Income'])
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Annual Income (k$)')
axes[2].set_title('Income vs Age')

plt.tight_layout()
plt.show()

**Interpretation**

The Income vs Spending plot shows the clearest customer groups. Approximately five visible groups can be identified: low income/low spending, low income/high spending, middle income/middle spending, high income/low spending, and high income/high spending.

The Age vs Spending and Age vs Income plots show weaker separation because the groups overlap more.

K-Means is unsupervised learning, so there is no true target label to compare against. Accuracy and a train/test split are therefore not appropriate. We evaluate clusters using distance, inertia, silhouette scores, plots, and domain knowledge.

**Questions** (answer with your own numbers)

(a) How many groups can you see by eye in each of the three plots? Which plot shows the groups most clearly?

(b) Why can we not use accuracy or a train/test split to judge a clustering result?



**(a)** In the Income vs Spending plot, approximately 5 groups are visible. In the Age vs Spending and Age vs Income plots, the groups are less clearly separated. The Income vs Spending plot shows the groups most clearly.

**(b)** We cannot use accuracy or a train/test split because clustering is unsupervised and has no true target labels. We evaluate clustering using inertia, silhouette scores, visual separation, and how meaningful the groups are.

---

## Part 1 — K-Means From Scratch (35 minutes)

In the lecture you did K-Means by hand. Here you turn your pen-and-paper steps into code, check your answers against the lecture slides, and then build your own K-Means function and compare it with scikit-learn. Using KMeans() inside your own function is not allowed.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| K | The number of clusters. You choose it before running. | `KMeans(n_clusters=5)` |
| Centroid | The centre (mean) of a cluster. It is a point, not always a real row. | `model.cluster_centers_` |
| Distance | 1 feature: |x - m|. 2 or more features: Euclidean, sqrt of the sum of squared differences. | `np.abs(a-b)  or  np.sqrt(((a-b)**2).sum(axis=1))` |
| Assign step | Give each point to its nearest centroid. | `d.argmin(axis=1)` |
| Update step | Move each centroid to the mean of the points assigned to it. | `X[labels==k].mean(axis=0)` |
| Convergence | Stop when no point changes cluster (centroids stop moving). | `np.allclose(new_c, c)` |
| Inertia | Sum of squared distances from every point to its own centroid. Smaller = tighter clusters. | `model.inertia_` |
| Broadcasting | NumPy trick: x[:, None] - m[None, :] gives every point-to-centroid difference in one line. | `x[:, None, :] - C[None, :, :]` |


### Task 1.1 — Lecture Example 1 in code (one feature)

Reproduce the 9-number example from the lecture (2, 4, 10, 12, 3, 20, 30, 11, 25) with K = 2 and starting centroids 4 and 11.

1. Write a loop that, in every iteration, computes the distances, assigns the clusters, updates the centroids and prints both clusters and the new centroids.
2. Stop when no centroid changes. Print the final centroids and the inertia.
3. Compare with the lecture slides (final centroids 7 and 25, inertia 150).

**Hints:**
- Distance matrix: np.abs(x[:, None] - m[None, :]) has shape (9, 2).
- Cluster of each point: dist.argmin(axis=1).


In [ ]:
# YOUR CODE HERE
x = np.array([2, 4, 10, 12, 3, 20, 30, 11, 25], dtype=float)
centroids = np.array([4.0, 11.0])

for iteration in range(1, 101):
    distances = np.abs(x[:, None] - centroids[None, :])
    labels = distances.argmin(axis=1)

    new_centroids = centroids.copy()
    for k in range(2):
        if np.any(labels == k):
            new_centroids[k] = x[labels == k].mean()

    print(f'Iteration {iteration}')
    for k in range(2):
        print(f'Cluster {k + 1}: {x[labels == k]}')
    print('New centroids:', new_centroids)

    if np.allclose(new_centroids, centroids):
        centroids = new_centroids
        break

    centroids = new_centroids

distances = np.abs(x[:, None] - centroids[None, :])
labels = distances.argmin(axis=1)
inertia = np.sum((x - centroids[labels]) ** 2)

print('Final centroids:', centroids)
print('Inertia:', inertia)

### Task 1.2 — Lecture Example 2 in code (two features)

Do the height/weight example: P1 (185, 72), P2 (170, 56), P3 (168, 60), P4 (179, 68), P5 (182, 72), P6 (188, 77). K = 2, start with P1 and P2.

1. Use Euclidean distance and the same loop idea. Print the cluster of every point and the centroids per iteration.
2. Print the final distance table and check one value by hand (the lecture shows d(P3, M2) = 4.47 in the first iteration).


In [ ]:
# YOUR CODE HERE
X_hw = np.array([
    [185, 72],
    [170, 56],
    [168, 60],
    [179, 68],
    [182, 72],
    [188, 77]
], dtype=float)

centroids_hw = X_hw[[0, 1]].copy()

for iteration in range(1, 101):
    distances = np.sqrt(
        ((X_hw[:, None, :] - centroids_hw[None, :, :]) ** 2).sum(axis=2)
    )
    labels_hw = distances.argmin(axis=1)

    new_centroids_hw = centroids_hw.copy()
    for k in range(2):
        if np.any(labels_hw == k):
            new_centroids_hw[k] = X_hw[labels_hw == k].mean(axis=0)

    print(f'Iteration {iteration}')
    for k in range(2):
        print(f'Cluster {k + 1}:', np.where(labels_hw == k)[0] + 1)
    print('Centroids:', new_centroids_hw)

    if np.allclose(new_centroids_hw, centroids_hw):
        centroids_hw = new_centroids_hw
        break

    centroids_hw = new_centroids_hw

final_distances_hw = np.sqrt(
    ((X_hw[:, None, :] - centroids_hw[None, :, :]) ** 2).sum(axis=2)
)

print('Final distance table:')
display(pd.DataFrame(
    final_distances_hw,
    index=[f'P{i}' for i in range(1, 7)],
    columns=['Centroid 1', 'Centroid 2']
))
print('Distance d(P3, M2) in first iteration:', np.sqrt(20))

### Task 1.3 — Write your own K-Means function

Write my_kmeans(X, init_centroids, max_iter=100) for any number of features.

1. Return four things: labels, final centroids, inertia and the number of iterations.
2. Test it on Example 1: it must give centroids 7 and 25 and inertia 150.
3. Think about what to do if a cluster ends up empty (keep its old centroid).


In [ ]:
# YOUR CODE HERE
def my_kmeans(X, init_centroids, max_iter=100):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    centroids = np.asarray(init_centroids, dtype=float).copy()

    for iteration in range(1, max_iter + 1):
        distances = np.sqrt(
            ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        )
        labels = distances.argmin(axis=1)

        new_centroids = centroids.copy()
        for k in range(len(centroids)):
            if np.any(labels == k):
                new_centroids[k] = X[labels == k].mean(axis=0)

        if np.allclose(new_centroids, centroids):
            centroids = new_centroids
            break

        centroids = new_centroids

    distances = np.sqrt(
        ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    )
    labels = distances.argmin(axis=1)
    inertia = np.sum((X - centroids[labels]) ** 2)

    return labels, centroids, inertia, iteration

In [ ]:
# YOUR CODE HERE
labels_1, centers_1, inertia_1, iterations_1 = my_kmeans(
    x, np.array([[4.0], [11.0]])
)

print('Clusters:')
for k in range(2):
    print(f'Cluster {k + 1}:', x[labels_1 == k])

print('Centroids:', centers_1.ravel())
print('Inertia:', inertia_1)
print('Iterations:', iterations_1)

### Task 1.4 — Check your function against scikit-learn

Run your function and KMeans on the same data with the same starting centroids and compare.

1. Use X2 = Income and Spending (200 x 2). Take the first 5 rows as the starting centroids.
2. Run my_kmeans. Run KMeans(n_clusters=5, init=<your 5 rows>, n_init=1, tol=0, algorithm='lloyd').
3. Compare iterations, inertia, labels and centroids.


In [ ]:
# YOUR CODE HERE
X2 = df[['Income', 'Spending']].values
initial_centroids = X2[:5].copy()

labels_custom, centers_custom, inertia_custom, iterations_custom = my_kmeans(
    X2, initial_centroids
)

model_custom_check = KMeans(
    n_clusters=5,
    init=initial_centroids,
    n_init=1,
    tol=0,
    algorithm='lloyd',
    max_iter=100
)

labels_sklearn = model_custom_check.fit_predict(X2)

print('Custom iterations:', iterations_custom)
print('Sklearn iterations:', model_custom_check.n_iter_)
print('Custom inertia:', inertia_custom)
print('Sklearn inertia:', model_custom_check.inertia_)
print('Labels match:', np.array_equal(labels_custom, labels_sklearn))
print('Centroids match:', np.allclose(centers_custom, model_custom_check.cluster_centers_))
print('Inertia matches:', np.isclose(inertia_custom, model_custom_check.inertia_))

**Questions** (answer with your own numbers)

(a) Example 1: the point 10 started in cluster C2 but moved to C1 in iteration 2. Explain why, using the centroid values.

(b) What makes the loop stop? Why is the stopping guaranteed even though the final answer is not always the best one?

(c) Did your function match scikit-learn exactly (labels, centroids, inertia)? If anything differed, explain why.

(d) What does a centroid mean for a group of customers? Why must the features be numeric for K-Means?



**Answers**

(a) Point 10 initially belongs to the second cluster. After the first update, the centroids become approximately 5.25 and 19.6. In the next assignment step, point 10 is closer to the first centroid than the second, so it changes clusters.

(b) The loop stops when the centroids no longer change. Each assign/update step does not increase inertia, and there are only finitely many possible assignments. The algorithm therefore converges to a stable solution, but it may be a local minimum rather than the global best solution.

(c) The custom function and scikit-learn should produce matching results when they use the same initialization and convergence settings. Small differences may occur because of stopping tolerances or implementation details.

(d) A centroid is the mean feature value of a cluster. K-Means requires numeric features because it calculates distances and averages. Categorical text must be encoded before clustering.

---

## Part 2 — Choosing K: Elbow and Silhouette (30 minutes)

The algorithm needs K before it starts, but nobody tells us K. The lecture gave two tools: the elbow (inertia) and the silhouette score. Here you use both, then calculate a silhouette by hand to prove you understand it.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Elbow method | Plot inertia for K = 2, 3, 4 ... Inertia always falls as K grows, so look for the K where the fall suddenly becomes small (the bend). | `model.inertia_ in a loop over K` |
| a | Average distance from a point to the other points in its own cluster (smaller = better). | `d[same_cluster].mean()` |
| b | Average distance from the point to the points of the nearest other cluster (larger = better). | `min over other clusters` |
| Silhouette s | s = (b - a) / max(a, b). Close to +1: well placed. Near 0: on the border. Negative: probably in the wrong cluster. | `silhouette_samples(X, labels)` |
| Average silhouette | The mean of s over all points. The K with the highest value is a good candidate. | `silhouette_score(X, labels)` |


### Task 2.1 — Elbow and silhouette for K = 2 to 10

Decide how many customer groups exist using Income and Spending (X2).

1. For K = 2 to 10 fit KMeans(n_clusters=K, n_init=10, random_state=SEED) and record the inertia and the average silhouette.
2. Show a table (K, Inertia, Silhouette) and two plots: elbow curve and silhouette bars.
3. Print the K with the highest silhouette.


In [ ]:
# YOUR CODE HERE
results_k = []

for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = model.fit_predict(X2)

    results_k.append({
        'K': k,
        'Inertia': model.inertia_,
        'Silhouette': silhouette_score(X2, labels)
    })

results_k_df = pd.DataFrame(results_k)
display(results_k_df.round(3))

best_k_silhouette = results_k_df.loc[
    results_k_df['Silhouette'].idxmax(), 'K'
]

print('Best K by silhouette:', best_k_silhouette)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    results_k_df['K'],
    results_k_df['Inertia'],
    marker='o'
)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].bar(
    results_k_df['K'],
    results_k_df['Silhouette']
)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Average Silhouette')
axes[1].set_title('Silhouette Scores')

plt.tight_layout()
plt.show()

**Interpretation**

Inertia decreases as K increases because adding clusters allows points to be closer to their centroids. The elbow plot shows a substantial improvement up to approximately K = 5, after which the improvement becomes smaller.

The highest average silhouette score is approximately 0.554 at K = 5. Both the elbow method and silhouette score support choosing five customer clusters.

### Task 2.2 — Silhouette by hand

Prove that you understand the formula by calculating it yourself.

1. Fit K = 5. Write a function that returns a, b and s for one point (use only NumPy distances). Apply it to customer 1 (row 0) and compare with silhouette_samples.
2. Show the count, mean and minimum silhouette of every cluster, and the number of customers with a negative silhouette.
3. Draw a silhouette plot of all customers sorted inside each cluster.


In [ ]:
# YOUR CODE HERE
model_k5 = KMeans(n_clusters=5, n_init=10, random_state=SEED)
labels_k5 = model_k5.fit_predict(X2)

def silhouette_manual(X, labels, index):
    point = X[index]
    own_cluster = labels[index]

    own_points = X[labels == own_cluster]
    own_distances = np.sqrt(((own_points - point) ** 2).sum(axis=1))

    if len(own_points) > 1:
        a = own_distances[own_distances > 0].mean()
    else:
        a = 0

    other_clusters = np.unique(labels[labels != own_cluster])
    b_values = []

    for cluster in other_clusters:
        other_points = X[labels == cluster]
        distances = np.sqrt(((other_points - point) ** 2).sum(axis=1))
        b_values.append(distances.mean())

    b = min(b_values)
    s = (b - a) / max(a, b)

    return a, b, s

a0, b0, s0 = silhouette_manual(X2, labels_k5, 0)
s0_sklearn = silhouette_samples(X2, labels_k5)[0]

print('Customer 1')
print('a:', a0)
print('b:', b0)
print('Manual silhouette:', s0)
print('Sklearn silhouette:', s0_sklearn)

In [ ]:
# YOUR CODE HERE
sil_values = silhouette_samples(X2, labels_k5)

silhouette_summary = pd.DataFrame({
    'Cluster': np.arange(5),
    'Count': [
        np.sum(labels_k5 == k) for k in range(5)
    ],
    'Mean Silhouette': [
        sil_values[labels_k5 == k].mean() for k in range(5)
    ],
    'Minimum Silhouette': [
        sil_values[labels_k5 == k].min() for k in range(5)
    ]
})

display(silhouette_summary.round(3))
print('Customers with negative silhouette:', np.sum(sil_values < 0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

y_lower = 10

for k in range(5):
    cluster_silhouette = np.sort(sil_values[labels_k5 == k])
    size = len(cluster_silhouette)
    y_upper = y_lower + size

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        cluster_silhouette
    )

    ax.text(-0.05, y_lower + size / 2, str(k))
    y_lower = y_upper + 10

ax.axvline(
    silhouette_score(X2, labels_k5),
    linestyle='--'
)
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot for K = 5')
plt.show()

**Interpretation**

Customer 1's manual silhouette should match the value returned by scikit-learn, apart from minor floating-point differences. A value close to +1 indicates that the customer fits its assigned cluster well; a value near 0 indicates a boundary customer; and a negative value suggests that another cluster may fit better.

The silhouette summary identifies the weakest-separated cluster through its lowest mean silhouette. Negative silhouettes indicate customers whose average distance to another cluster is smaller than their average distance to their own cluster.

### Task 2.3 — The lecture practice problem

Points 1, 3, 5, 7, 11, 13, 15 with K = 2 and starting centroids 1 and 11 (the 'Practice' slide).

1. Run your my_kmeans on these points and print the final clusters, centroids and iterations.
2. Calculate a, b and s for the points 1, 7 and 11 with your function and compare with silhouette_samples.
3. Also write the iterations on paper (clusters and centroids after each iteration) and keep the photo or notes for your own revision.


In [ ]:
# YOUR CODE HERE
practice = np.array([1, 3, 5, 7, 11, 13, 15], dtype=float)

practice_labels, practice_centers, practice_inertia, practice_iterations = my_kmeans(
    practice, np.array([[1.0], [11.0]])
)

print('Clusters:')
for k in range(2):
    print(f'Cluster {k + 1}:', practice[practice_labels == k])

print('Centroids:', practice_centers.ravel())
print('Inertia:', practice_inertia)
print('Iterations:', practice_iterations)

practice_X = practice.reshape(-1, 1)
practice_sil = silhouette_samples(practice_X, practice_labels)

for point in [1, 7, 11]:
    index = np.where(practice == point)[0][0]
    a, b, s = silhouette_manual(practice_X, practice_labels, index)
    print(f'Point {point}: a={a:.3f}, b={b:.3f}, manual={s:.3f}, sklearn={practice_sil[index]:.3f}')

**Questions** (answer with your own numbers)

(a) Which K does the elbow suggest and which K does the silhouette suggest? Do they agree? Quote the numbers (for example how much the inertia drops from K=4 to 5 and from 5 to 6).

(b) What is the silhouette of customer 1 and what does it tell you? Which cluster has the weakest average silhouette and how many customers have a negative value?

(c) In the practice problem, is point 1 or point 7 less well placed? Explain using a and b.

(d) Why should we never choose K only by the lowest inertia?



**Answers**

(a) The elbow suggests K = 5 because the inertia reduction becomes smaller after that point. The silhouette also suggests K = 5 because it has the highest average score, approximately 0.554. The two methods agree.

(b) Customer 1's silhouette is the value printed above. A positive value means the customer is closer, on average, to its own cluster than to the nearest alternative cluster. The cluster with the lowest mean silhouette is the weakest-separated group; the code prints its mean and the total number of negative silhouettes.

(c) Point 7 is less well placed if its silhouette is lower than point 1's. Compare their a and b values: a is the average distance within its own cluster, while b is the average distance to the nearest other cluster.

(d) Lowest inertia alone always favors increasing K. With one cluster per point, inertia can become zero, but the result is not a useful segmentation. We must balance compactness with meaningful separation and interpretability.

---

## Part 3 — Starting Points and Stability (20 minutes)

The lecture said 'starting points matter - run several starts and keep the best'. You will now measure how much, using the same data and the same K.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Local minimum | A solution that cannot be improved by the assign/update loop but is not the best possible one. | `A higher final inertia than the best run.` |
| init='random' | Pick K random rows as the starting centroids. | `KMeans(init='random')` |
| init='k-means++' | Pick starting centroids that are far from each other. This is the default and is usually better. | `KMeans(init='k-means++')` |
| n_init | How many different starts to try. scikit-learn keeps the run with the lowest inertia. | `KMeans(n_init=10)` |
| Adjusted Rand Index (ARI) | Agreement between two groupings. 1 = same groups, about 0 = no better than random. It ignores the cluster numbers (cluster 2 in one run can be cluster 4 in another). | `adjusted_rand_score(a, b)` |


### Task 3.1 — 30 different seeds

Measure how often K-Means finds the best answer.

1. With K = 5 on X2, run KMeans 30 times (random_state = 0 to 29) in three settings: init='random' with n_init=1, k-means++ with n_init=1, k-means++ with n_init=10.
2. Build a table: setting, number of distinct final inertia values (round to 1 decimal), lowest, highest, and the percentage of runs that reached the best inertia.
3. Draw one histogram of the final inertia values for the three settings.


In [ ]:
# YOUR CODE HERE
settings = [
    ('Random, n_init=1', 'random', 1),
    ('K-means++, n_init=1', 'k-means++', 1),
    ('K-means++, n_init=10', 'k-means++', 10)
]

stability_runs = []

for setting, init, n_init in settings:
    for seed in range(30):
        model = KMeans(
            n_clusters=5,
            init=init,
            n_init=n_init,
            random_state=seed
        )
        model.fit(X2)

        stability_runs.append({
            'Setting': setting,
            'Seed': seed,
            'Inertia': model.inertia_,
            'Model': model
        })

stability_df = pd.DataFrame(stability_runs)

stability_summary = stability_df.groupby('Setting').agg(
    Distinct_Inertias=('Inertia', lambda x: x.round(1).nunique()),
    Lowest=('Inertia', 'min'),
    Highest=('Inertia', 'max')
)

best_inertia = stability_df.groupby('Setting')['Inertia'].transform('min')
stability_df['Reached_Best'] = np.isclose(
    stability_df['Inertia'], best_inertia
)

best_percent = stability_df.groupby('Setting')['Reached_Best'].mean() * 100
stability_summary['Best_Run_Percent'] = best_percent

display(stability_summary.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for setting, _, _ in settings:
    values = stability_df.loc[
        stability_df['Setting'] == setting, 'Inertia'
    ]
    ax.hist(values, alpha=0.5, label=setting)

ax.set_xlabel('Final Inertia')
ax.set_ylabel('Frequency')
ax.set_title('K-Means Inertia Across 30 Seeds')
ax.legend()
plt.show()

### Task 3.2 — Best run vs worst run

See what a bad local minimum looks like.

1. From the random-init runs take the best and the worst. Draw both clusterings with their centroids side by side.
2. Print their inertia and their ARI.


In [ ]:
# YOUR CODE HERE
random_runs = stability_df[
    stability_df['Setting'] == 'Random, n_init=1'
]

best_run = random_runs.loc[random_runs['Inertia'].idxmin()]
worst_run = random_runs.loc[random_runs['Inertia'].idxmax()]

best_model = best_run['Model']
worst_model = worst_run['Model']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, title in [
    (axes[0], best_model, 'Best Random Initialization'),
    (axes[1], worst_model, 'Worst Random Initialization')
]:
    ax.scatter(
        X2[:, 0],
        X2[:, 1],
        c=model.labels_
    )
    ax.scatter(
        model.cluster_centers_[:, 0],
        model.cluster_centers_[:, 1],
        marker='X',
        s=200
    )
    ax.set_xlabel('Income')
    ax.set_ylabel('Spending')
    ax.set_title(title)

plt.tight_layout()
plt.show()

print('Best inertia:', best_model.inertia_)
print('Worst inertia:', worst_model.inertia_)
print('ARI:', adjusted_rand_score(best_model.labels_, worst_model.labels_))

**Interpretation**

The best run has tighter clusters and lower inertia than the worst run. The ARI measures agreement between the two clusterings without requiring their cluster numbers to match. An ARI near 1 means the groupings are almost identical; a value near 0 indicates little agreement beyond chance.

**Questions** (answer with your own numbers)

(a) How many different final inertia values did the random-start setting produce, and what percentage of its runs reached the best one? Compare with the other two settings.

(b) Why can the same data and the same K give different answers?

(c) How do k-means++ and n_init help? What did n_init=10 give in your table?

(d) Why do we use ARI to compare two clusterings instead of comparing the label numbers directly?



**Answers**

(a) The number of distinct random-start inertia values and the percentage of runs reaching the best result are shown in the summary table. Compare these with the k-means++ settings to evaluate stability.

(b) K-Means can reach different local minima because the initial centroids affect the assignment and update steps. Different starts can lead to different final partitions.

(c) K-means++ selects more spread-out starting points, reducing poor initializations. n_init=10 runs the algorithm multiple times and keeps the lowest-inertia solution.

(d) Cluster labels are arbitrary. The same grouping may be labeled 0, 1, 2 in one run and 3, 4, 1 in another. ARI compares the actual grouping structure rather than the numeric labels.

---

## Part 4 — Scaling: Units Decide the Clusters (25 minutes)

The lecture warned: 'Rs 5000 would swamp 3 visits'. You will prove it twice: once by only changing the unit of Income, and once on the Wine data where the features have very different sizes and the true groups are known.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Feature scaling | Putting all features on a similar scale so that no feature wins the distance only because its numbers are bigger. | `StandardScaler().fit_transform(X)` |
| StandardScaler | z = (x - mean) / std. Every column gets mean 0 and standard deviation 1. | `StandardScaler()` |
| External check | Comparing clusters with known true groups. Possible only for teaching data like Wine. Real clustering problems have no labels. | `adjusted_rand_score(y_true, labels)` |
| PCA (Lab 8) | Reduces many features to 2 so we can draw the data. Used here only for the plot. | `PCA(n_components=2)` |


### Task 4.1 — Change the unit, change the clusters

Use Age and Income with K = 4. Income is in thousands of dollars (15 to 137). Make a second copy where Income is in dollars (multiply by 1000).

1. Cluster four versions: raw k$, raw dollars, scaled k$, scaled dollars (StandardScaler, n_init=10, random_state=SEED).
2. Print ARI between: raw k$ vs raw dollars, scaled k$ vs scaled dollars, raw dollars vs scaled dollars.
3. Draw Age vs Income (in k$) coloured by the clusters of raw k$, raw dollars and scaled.


In [ ]:
# YOUR CODE HERE
X_age_income = df[['Age', 'Income']].values
X_age_income_dollars = X_age_income.copy()
X_age_income_dollars[:, 1] *= 1000

X_age_income_scaled = StandardScaler().fit_transform(X_age_income)
X_age_income_dollars_scaled = StandardScaler().fit_transform(
    X_age_income_dollars
)

models_unit = {}

for name, X in [
    ('Raw k$', X_age_income),
    ('Raw dollars', X_age_income_dollars),
    ('Scaled k$', X_age_income_scaled),
    ('Scaled dollars', X_age_income_dollars_scaled)
]:
    model = KMeans(n_clusters=4, n_init=10, random_state=SEED)
    models_unit[name] = model.fit(X)

print('ARI raw k$ vs raw dollars:',
      adjusted_rand_score(models_unit['Raw k$'].labels_,
                         models_unit['Raw dollars'].labels_))

print('ARI scaled k$ vs scaled dollars:',
      adjusted_rand_score(models_unit['Scaled k$'].labels_,
                         models_unit['Scaled dollars'].labels_))

print('ARI raw dollars vs scaled dollars:',
      adjusted_rand_score(models_unit['Raw dollars'].labels_,
                         models_unit['Scaled dollars'].labels_))

### Task 4.2 — Wine data: raw vs scaled

Load the Wine data with load_wine(): 178 wines, 13 chemical measurements, 3 true grape cultivars. The true cultivar is used ONLY to check the result, never to fit K-Means.

1. Print the three largest and the smallest standard deviation of the features.
2. Run KMeans(K=3, n_init=10, random_state=SEED) on the raw data and on the scaled data. For each report ARI against the true cultivar, silhouette score and cluster sizes.


In [ ]:
# YOUR CODE HERE
wine_data = load_wine()
X_wine = wine_data.data
y_wine = wine_data.target
wine_features = wine_data.feature_names

wine_df = pd.DataFrame(X_wine, columns=wine_features)

wine_std = wine_df.std().sort_values(ascending=False)

print('Three largest standard deviations:')
display(wine_std.head(3))

print('Smallest standard deviation:')
display(wine_std.tail(1))

X_wine_scaled = StandardScaler().fit_transform(X_wine)

wine_raw_model = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=SEED
).fit(X_wine)

wine_scaled_model = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=SEED
).fit(X_wine_scaled)

wine_results = pd.DataFrame({
    'Data': ['Raw', 'Scaled'],
    'ARI': [
        adjusted_rand_score(y_wine, wine_raw_model.labels_),
        adjusted_rand_score(y_wine, wine_scaled_model.labels_)
    ],
    'Silhouette': [
        silhouette_score(X_wine, wine_raw_model.labels_),
        silhouette_score(X_wine_scaled, wine_scaled_model.labels_)
    ],
    'Cluster Sizes': [
        np.bincount(wine_raw_model.labels_),
        np.bincount(wine_scaled_model.labels_)
    ]
})

display(wine_results)

### Task 4.3 — Draw it

Compare the three views in one picture.

1. Use one PCA of the scaled data and draw three plots with the same axes: true cultivars, raw clusters, scaled clusters.


In [ ]:
# YOUR CODE HERE
pca = PCA(n_components=2)
X_wine_pca = pca.fit_transform(X_wine_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_data = [
    (y_wine, 'True Cultivars'),
    (wine_raw_model.labels_, 'Raw K-Means'),
    (wine_scaled_model.labels_, 'Scaled K-Means')
]

for ax, (labels, title) in zip(axes, plot_data):
    ax.scatter(
        X_wine_pca[:, 0],
        X_wine_pca[:, 1],
        c=labels
    )
    ax.set_xlabel('Principal Component 1')
    ax.set_ylabel('Principal Component 2')
    ax.set_title(title)

plt.tight_layout()
plt.show()

**Questions** (answer with your own numbers)

(a) What happened when only the unit of Income changed? Explain it with the distance formula (which feature controls the distance now?).

(b) On the Wine data which feature dominated the raw distance and how big is its standard deviation compared with the others? What were the ARI values raw vs scaled?

(c) The silhouette was higher on the raw Wine data although the clustering was worse. Explain why a silhouette can mislead here and why comparing it between raw and scaled data is not fair.

(d) In a real project with no labels, how will you decide whether to scale? Write your rule in one or two sentences.



**Answers**

(a) Changing Income to dollars makes Income dominate the distance because its squared difference becomes one million times larger. The clusters are then driven mainly by Income rather than Age.

(b) Proline dominates the raw Wine distance. Its standard deviation is much larger than the standard deviations of most other features. The raw and scaled ARI values are printed in the Wine results table.

(c) Silhouette measures separation in the space used to calculate distances, not agreement with true cultivars. Raw features can form compact groups around a dominant feature while still failing to recover the real classes. Scaling changes the distance geometry, so the two silhouette scores are not a fair direct comparison.

(d) I would scale when features have different units or substantially different ranges and when each feature should contribute meaningfully. I would retain raw units only when their relative magnitudes are intentional and justified by the problem.

---

## Part 5 — Outliers and Anomalies (20 minutes)

The lecture said 'one extreme point drags a centroid'. You will add one extreme customer, see the damage, and then catch it with a distance rule (the anomaly detection idea from the lecture).

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Outlier | A point far away from all the others. K-Means uses means and squared distances, so one outlier has a big effect. | `A very large distance to the nearest centroid.` |
| Distance to own centroid | How far a point is from the centre of its cluster. Large = the point does not fit any group well. | `model.transform(X).min(axis=1)` |
| mean + 3 std rule | Flag a point if its distance is larger than the mean distance plus 3 standard deviations. | `d > d.mean() + 3*d.std()` |
| Anomaly detection | Using the clusters to find strange points (far from every centroid). | `Sort by the distance column.` |


### Task 5.1 — One extreme customer

Add one extra row to X2: Income = 400, Spending = 99. Use K = 5 on Income and Spending.

1. Fit KMeans (n_init=10, random_state=SEED) without and with the extra customer.
2. Print the cluster sizes, the centroids (sorted by income) and the ARI between the two results on the original 200 customers.
3. Draw the two clusterings side by side.


In [ ]:
# YOUR CODE HERE
X2_outlier = np.vstack([X2, [400, 99]])

model_original = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
).fit(X2)

model_with_outlier = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
).fit(X2_outlier)

print('Original cluster sizes:', np.bincount(model_original.labels_))
print('With outlier cluster sizes:', np.bincount(model_with_outlier.labels_))

centers_original = pd.DataFrame(
    model_original.cluster_centers_,
    columns=['Income', 'Spending']
).sort_values('Income')

centers_outlier = pd.DataFrame(
    model_with_outlier.cluster_centers_,
    columns=['Income', 'Spending']
).sort_values('Income')

print('Original centroids:')
display(centers_original)

print('Centroids with outlier:')
display(centers_outlier)

print('ARI on original customers:',
      adjusted_rand_score(
          model_original.labels_,
          model_with_outlier.predict(X2)
      ))

In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    X2[:, 0], X2[:, 1],
    c=model_original.labels_
)
axes[0].scatter(
    model_original.cluster_centers_[:, 0],
    model_original.cluster_centers_[:, 1],
    marker='X', s=200
)
axes[0].set_title('Without Outlier')

axes[1].scatter(
    X2_outlier[:, 0], X2_outlier[:, 1],
    c=model_with_outlier.labels_
)
axes[1].scatter(
    model_with_outlier.cluster_centers_[:, 0],
    model_with_outlier.cluster_centers_[:, 1],
    marker='X', s=200
)
axes[1].set_title('With Outlier')

for ax in axes:
    ax.set_xlabel('Income')
    ax.set_ylabel('Spending')

plt.tight_layout()
plt.show()

### Task 5.2 — Catch it and repair

Use the model fitted on the original 200 customers.

1. Compute every customer's distance to its nearest centroid, and the distance of the extra customer to the nearest centroid of the same model.
2. Apply the rule mean + 3 std (mean and std from the 200 real customers). How many real customers are flagged? Is the extra customer flagged?
3. Show the 5 real customers that are farthest from their centroid (CustomerID, Age, Income, Spending, Cluster, Distance).
4. Remove all flagged points, refit K = 5 and report the cluster sizes and the ARI against the original model on the 200 customers (use predict).


In [ ]:
# YOUR CODE HERE
distances_original = model_original.transform(X2)
nearest_distances = distances_original.min(axis=1)

extra_distance = model_original.transform(
    np.array([[400, 99]])
).min()

threshold = nearest_distances.mean() + 3 * nearest_distances.std()
flagged_mask = nearest_distances > threshold

print('Mean distance:', nearest_distances.mean())
print('Standard deviation:', nearest_distances.std())
print('Threshold:', threshold)
print('Flagged real customers:', flagged_mask.sum())
print('Extra customer distance:', extra_distance)
print('Extra customer flagged:', extra_distance > threshold)
df_anomaly = df.copy()
df_anomaly['Cluster'] = model_original.labels_
df_anomaly['Distance'] = nearest_distances

print('Five farthest real customers:')
display(
    df_anomaly.nlargest(5, 'Distance')[
        ['CustomerID', 'Age', 'Income', 'Spending', 'Cluster', 'Distance']
    ]
)
df_clean = df.loc[~flagged_mask].copy()
X_clean = df_clean[['Income', 'Spending']].values

model_clean = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
).fit(X_clean)

clean_predictions = model_clean.predict(X2)

print('Cleaned cluster sizes:', np.bincount(model_clean.labels_))
print('ARI against original model:',
      adjusted_rand_score(model_original.labels_, clean_predictions))

**Questions** (answer with your own numbers)

(a) What did the single extreme customer do to the clusters? Which real groups were affected? Use sizes and centroids.

(b) How well did the rule detect the extra customer? The rule also flagged some real customers: are they mistakes in the data or something worth looking at? Look at their Income and Spending.

(c) Name two ways to deal with outliers and one risk of simply deleting them.



**Answers**

(a) The extreme customer can pull a centroid toward high Income and Spending, changing the sizes and positions of the original high-income groups. Use the printed cluster sizes and centroids to identify which groups changed most.

(b) The rule detects the extra customer if its distance exceeds the calculated threshold. Real customers flagged by the rule may be legitimate unusual customers, so their Income and Spending should be inspected before removal.

(c) Two ways to handle outliers are to investigate and correct data errors, or use a robust method such as clustering after excluding confirmed anomalies. Deleting valid unusual customers can remove important business information and bias the segmentation.

---

## Part 6 — From Clusters to Customer Segments (25 minutes)

Clusters are numbers (0, 1, 2 ...). The business needs names and actions. Cluster numbers are arbitrary labels: cluster 3 in one run has no meaning by itself. Use the original 200 customers and K = 5 on Income and Spending (random_state=SEED).

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Cluster profile | The average of every feature inside each cluster, plus the cluster size. This is how you read what a cluster means. | `df.groupby('Cluster').agg(...)` |
| Segment | A cluster with a name and an action, for example 'High income / Low spending'. | `df['Cluster'].map(names)` |
| Crosstab | Counts or percentages of one category inside another (Gender inside Segment). | `pd.crosstab(a, b, normalize='index')` |


### Task 6.1 — Profile and name the segments

Make the clustering readable for a manager.

1. Add the cluster number to the data. Build a profile table: number of customers, mean Age, mean Income, mean Spending and % female for each cluster, sorted by Income.
2. Create segment names with a small rule (Income below 40 = Low, 70 or more = High, otherwise Mid; Spending below 35 = Low, 65 or more = High, otherwise Mid). Print the size of each segment.


In [ ]:
# YOUR CODE HERE
model_segments = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
).fit(X2)

df_segments = df.copy()
df_segments['Cluster'] = model_segments.labels_

profile = df_segments.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Mean_Age=('Age', 'mean'),
    Mean_Income=('Income', 'mean'),
    Mean_Spending=('Spending', 'mean'),
    Percent_Female=('Gender', lambda x: (x == 'Female').mean() * 100)
).sort_values('Mean_Income')

display(profile.round(2))

In [ ]:
# YOUR CODE HERE
def make_segment(row):
    income = 'Low' if row['Income'] < 40 else (
        'High' if row['Income'] >= 70 else 'Mid'
    )

    spending = 'Low' if row['Spending'] < 35 else (
        'High' if row['Spending'] >= 65 else 'Mid'
    )

    return f'{income} Income / {spending} Spending'

cluster_means = df_segments.groupby('Cluster')[['Income', 'Spending']].mean()

segment_names = {
    cluster: make_segment(row)
    for cluster, row in cluster_means.iterrows()
}

df_segments['Segment'] = df_segments['Cluster'].map(segment_names)

print('Segment sizes:')
display(df_segments['Segment'].value_counts())

### Task 6.2 — Does Gender matter? Does Age matter?

Gender and Age were not used in the clusters. Check whether they would add something.

1. Show the % of women in each segment and the overall %.
2. Scale Age, Income and Spending together and print the average silhouette for K = 2 to 8.


In [ ]:
# YOUR CODE HERE
gender_segment = pd.crosstab(
    df_segments['Segment'],
    df_segments['Gender'],
    normalize='index'
) * 100

print('Gender percentages by segment:')
display(gender_segment.round(2))

print('Overall gender percentages:')
display((df_segments['Gender'].value_counts(normalize=True) * 100).round(2))
X3 = df[['Age', 'Income', 'Spending']].values
X3_scaled = StandardScaler().fit_transform(X3)

silhouette_age_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=SEED
    )
    labels = model.fit_predict(X3_scaled)

    silhouette_age_results.append({
        'K': k,
        'Silhouette': silhouette_score(X3_scaled, labels)
    })

silhouette_age_df = pd.DataFrame(silhouette_age_results)
display(silhouette_age_df.round(3))

### Task 6.3 — A three-feature alternative

Cluster the scaled Age, Income and Spending with the best K from the table above.

1. Fit it, build its profile table (size, Age, Income, Spending) sorted by Age.
2. Print the ARI between this segmentation and your 2-feature K = 5 segmentation.


In [ ]:
# YOUR CODE HERE
best_k_age = int(
    silhouette_age_df.loc[
        silhouette_age_df['Silhouette'].idxmax(), 'K'
    ]
)

model_3features = KMeans(
    n_clusters=best_k_age,
    n_init=10,
    random_state=SEED
).fit(X3_scaled)

df_3features = df.copy()
df_3features['Cluster'] = model_3features.labels_

profile_3features = df_3features.groupby('Cluster').agg(
    Customers=('CustomerID', 'count'),
    Mean_Age=('Age', 'mean'),
    Mean_Income=('Income', 'mean'),
    Mean_Spending=('Spending', 'mean')
).sort_values('Mean_Age')

display(profile_3features.round(2))

print('Best K for three-feature clustering:', best_k_age)
print('ARI against two-feature K=5 segmentation:',
      adjusted_rand_score(
          model_segments.labels_,
          model_3features.labels_
      ))

**Questions** (answer with your own numbers)

(a) Give each of the 5 segments a short business name and one concrete action (an offer or a message). Use the numbers from your profile table.

(b) Which segment is the most valuable now and which one is the biggest opportunity? Why?

(c) Does Gender help to separate the segments? Quote the percentages.

(d) Does adding Age change the story? Which segmentation would you present and why? (Careful: silhouette values from different feature sets are not directly comparable.)



**Answers**

(a) Use the profile table to assign names and actions to the five clusters. For example, high-income/high-spending customers can receive loyalty rewards, while high-income/low-spending customers can receive personalized product recommendations.

(b) The most valuable current segment is generally the high-income/high-spending group because it combines purchasing power with strong spending behavior. The biggest opportunity may be a high-income/low-spending group because targeted offers could increase its spending.

(c) Gender helps separate segments only if the female percentages differ meaningfully across segments. Use the gender table to support the conclusion.

(d) Adding Age changes the feature space and may change cluster membership. I would present the segmentation whose profiles are clearest and whose actions are most useful to the marketing team. Silhouette scores from different feature sets are not directly comparable.

---

## Part 7 — Final Output and Wrap-Up (15 minutes)

Package your result like you would give it to a marketing team.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Deliverable | A file someone else can use without running your notebook. | `df.to_csv('customer_segments.csv', index=False)` |


### Task 7.1 — Save the segments

Create a CSV with CustomerID, Age, Gender, Income, Spending, Cluster and Segment.

1. Save it as customer_segments.csv and print the number of customers per segment.
2. Write a 6-8 line Executive Summary (Markdown): the K you chose and the evidence for it, the segments and their sizes, one action per segment (short), how you handled scaling and outliers, and one limitation.


In [ ]:
# YOUR CODE HERE
final_output = df_segments[
    ['CustomerID', 'Age', 'Gender', 'Income', 'Spending', 'Cluster', 'Segment']
].copy()

final_output.to_csv('customer_segments.csv', index=False)

print('Customers per segment:')
display(final_output['Segment'].value_counts())

print('Saved customer_segments.csv')

**Executive Summary** (6-8 lines)

- I selected K = 5 for the Income and Spending segmentation, supported by the elbow method and the highest silhouette score.
- The five clusters represent distinct combinations of customer income and spending behavior.
- The cluster profile table provides the size, average age, income, spending, and gender composition of each segment.
- Each segment can be targeted with a different marketing action based on its profile.
- I used StandardScaler when features had different units or scales.
- I investigated the effect of an extreme customer and applied a distance-based anomaly rule.
- A limitation is that K-Means assumes compact, approximately spherical clusters and is sensitive to outliers and feature scaling.


**(a) What was the most surprising result in this lab?**

The most surprising result was how strongly changing the unit of Income or adding a single extreme customer could change the K-Means clusters. This showed that scaling and outlier handling are essential for meaningful clustering.

**(b) If you had one more hour, what would you try next?**

I would compare the K-Means segments with hierarchical clustering and DBSCAN, investigate the customers flagged as anomalies, and test whether the segments remain stable on new customer data.

---